# CasDsl — a categorically organized CAS in Lean 4

This notebook walks through the CasDsl surface: a computer algebra system
whose operations are organized by the mathematical categories where they
first make sense. Every cell is ordinary Lean 4, elaborated by a persistent
worker. **Backend-blind syntax:** no expression ever names Sage, GAP, or an
algorithm — the routing layer selects implementations after the mathematical
operation is resolved.

We'll start with trusted arithmetic, then factorization, polynomials,
algebraic numbers, linear algebra, and calculus — and end with a deliberate
capability gap that shows what happens when a method is semantically
available but no backend has registered an implementation yet.

## 1 · Trusted arithmetic and assertions

`assert` is an operational assertion in the ordinary CAS sense: the
predicate is computed and trusted. Only `true` lets the cell commit — a
false or unknown result is a cell error, and the notebook state rolls back.
No Lean theorem is generated; this is a CAS, not a proof obligation
machine.

We start with the simplest possible assertion — integer arithmetic — then
move to modular arithmetic.

In [1]:
assert 2 + 3 = 5

Starting Lean worker (/tmp/journey-proof-70d28d7/qualification/lean-cas-dsl)…


1:0: ✓ 2 + 3 = 5


The `in ℤ/5` suffix changes the ring the assertion is evaluated in.
`2 + 3` is 5 in ℤ, but 5 ≡ 0 in ℤ/5, so the assertion holds.

In [2]:
assert 2 + 3 = 0 in ℤ/5

1:0: ✓ 2 + 3 = 0 in ℤ/5


## 2 · Factorization

`factor` is declared on the category of factorization-domain elements.
An integer receives it because `EuclideanElems(ℤ) ≤ FactorizationElems(ℤ)`
— the method arrives by functor composition, not by leaf-specific
forwarding code. The computation is routed to Sage, but the expression
`n.factor()` never names it.

We bind `n := 360` as an integer, then ask for its factorization.

In [3]:
let n := 360 in ℤ

1:0: n := 360 ∈ ℤ


In [4]:
n.factor()

1:0: 2^3 * 3^2 * 5


2^3 * 3^2 * 5

`gcd` works the same way — it's a method on the category of GCD-domain
elements, and integers inherit it through the same subcategory chain.

In [5]:
assert gcd(84, 30) = 6

1:0: ✓ gcd(84, 30) = 6


## 3 · Polynomials

Polynomial rings are first-class categories. We'll define
$p(x) = x^3 - 2x + 1$ in $\mathbb{Z}[x]$ and explore what the system can
tell us about it.

The `let p(x) := … in ℤ[x]` syntax binds `p` as a polynomial over the
integers. The `(x)` after the name tells the parser this is a polynomial
binder — `x` becomes the indeterminate.

In [6]:
let p(x) := x^3 - 2x + 1 in ℤ[x]

1:0: p := x^3 - 2x + 1 ∈ ℤ[x]


`p.deg()` returns the degree. This is a category-owned method — it's
declared on the category of polynomials and inherited by every concrete
polynomial ring.

In [7]:
p.deg()

1:0: 3


3

`p.roots()` asks for the roots of $p$ *in its coefficient ring*.
Since $p \in \mathbb{Z}[x]$, this asks for **integer roots** — and
$p(1) = 1 - 2 + 1 = 0$, so $1$ is one. The answer is `{1}`, not the empty
set.

The note the system prints is doing real mathematics: $x^3 - 2x + 1$
factors as $(x-1)(x^2 + x - 1)$, and $x^2 + x - 1$ has no integer roots,
so $p$ **does not split in ℤ** — its remaining two roots live in an
extension. The note states the multiplicities and names the explicit
`map p to ℂ[x]` that would reach the other roots. No root is silently
hidden.

In [8]:
p.roots()

1:0: p does not split in ℤ: its 1 root(s) there carry total multiplicity 1 of degree 3 (`p.factor()` shows the multiplicities), and the remaining 2 lie in an extension. For all of them: `let pC := map p to ℂ[x]`, then `pC.roots()`


1:0: {1}


{1}

The natural next step is to move $p$ to $\mathbb{Q}[x]$ along the
canonical inclusion $\mathbb{Z} \subseteq \mathbb{Q}$ — coefficient by
coefficient, via the registered canonical map. `map … to …` performs the
move, and the `let` binds the result so we can name it.

In [9]:
let pQ := map p to ℚ[x]

1:0: pQ := x^3 - 2x + 1 ∈ ℚ[x]


The root set does not change: $1$ was already an integer, and the
quadratic factor still has no rational roots — so `pQ.roots()` is `{1}`
again, and the note reports the same split, now relative to ℚ.

In [10]:
pQ.roots()

1:0: pQ does not split in ℚ: its 1 root(s) there carry total multiplicity 1 of degree 3 (`pQ.factor()` shows the multiplicities), and the remaining 2 lie in an extension. For all of them: `let pQC := map pQ to ℂ[x]`, then `pQC.roots()`


1:0: {1}


{1}

Evaluation at a point is ordinary substitution — `pQ(1) = 0` confirms
$1$ is a root of $p$ regardless of the ring its coefficients are read in.

In [11]:
assert pQ(1) = 0

1:0: ✓ pQ(1) = 0


The empty root set is also an answer, when it is the truth. SPEC.md's
$q(x) = x^2 - 2 \in \mathbb{Q}[x]$ has **no rational root** — $\sqrt{2}$ is
irrational — so `q.roots()` is `{}`: a correct result, not an error, and
not a silent reach into an extension field.

In [12]:
let q := x ↦ x² - 2 in ℚ[x]

1:0: q := x^2 - 2 ∈ ℚ[x]


In [13]:
q.roots()

1:0: q does not split in ℚ: its 0 root(s) there carry total multiplicity 0 of degree 2 (`q.factor()` shows the multiplicities), and the remaining 2 lie in an extension. For all of them: `let qC := map q to ℂ[x]`, then `qC.roots()`


1:0: {}


{}

## 4 · Exact algebraic numbers

CasDsl works with exact algebraic numbers — never decimals unless you
explicitly request an approximation. The ⊆-chain
$\mathbb{N} \subseteq \mathbb{Z} \subseteq \mathbb{Q} \subseteq \mathbb{R} \subseteq \mathbb{C}$
is read off the canonical-map registry, so membership and transport
between these systems is automatic.

We'll bind $z = 2 + 2i$ in ℂ and verify its absolute value.

In [14]:
let z := 2 + 2i in ℂ

1:0: z := 2 + 2i ∈ ℂ


$|2 + 2i| = \sqrt{2^2 + 2^2} = \sqrt{8} = 2\sqrt{2}$. The system
returns the exact surd, not $2.828\ldots$.

In [15]:
assert |z| = 2√2

1:0: √ denotes ONE root by convention — the non-negative branch (upward, i·√|d|, for a negative radicand): an embedding choice made by the spelling, logged at use (#31 item 10)


1:0: ✓ |z| = 2√2


Numerical approximation is an operation **on** an exact value, not a
replacement for it. `map √2 to ℝ/O(1/10^{10})` asks for $\sqrt{2}$
approximated to within $10^{-10}$ in ℝ. The tolerance is a request, not a
quotient — the underlying value remains exact.

In [16]:
map √2 to ℝ/O(1/10^{10})

1:0: √ denotes ONE root by convention — the non-negative branch (upward, i·√|d|, for a negative radicand): an embedding choice made by the spelling, logged at use (#31 item 10)


1:0: 1.4142135623 + O(1/10^{10})


1.4142135623 + O(1/10^{10})

## 5 · Linear algebra

Exact matrix arithmetic over ℚ. Matrix literals use **row-semicolon
syntax**: `[1, 2; 3, 4]` is a $2 	imes 2$ matrix whose rows are `[1, 2]`
and `[3, 4]`. We'll define one, compute its determinant, and invert it —
all exactly, with rational entries.

In [17]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

1:0: M := [1, 2; 3, 4] ∈ Mat₂(ℚ)


$\det(M) = 1 \cdot 4 - 2 \cdot 3 = -2$. The result is exact — no
floating-point.

In [18]:
assert M.det() = -2

1:0: ✓ M.det() = -2


$M^{-1} = \frac{1}{-2} egin{bmatrix} 4 & -2 \\ -3 & 1 \end{bmatrix}
= egin{bmatrix} -2 & 1 \\ 3/2 & -1/2 \end{bmatrix}$. Again, exact
rational entries.

In [19]:
M⁻¹

1:0: [-2, 1; 3/2, -1/2]


[-2, 1; 3/2, -1/2]

## 6 · Calculus

CasDsl distinguishes the **universal differential** $d(f)$ — a 1-form —
from the **derivation** $(d/dx)(f)$ — a polynomial. They are different
types and not equal, even when their coefficients match.

The indefinite integral $\int f\,dx$ returns the **coset** of
antiderivatives: $x^3 + x^2/2 + x + \mathbb{Q}$ means "the set of all
functions of the form $x^3 + x^2/2 + x + c$ where $c \in \mathbb{Q}$."

We'll work with $f(x) = 3x^2 + x + 1$ over ℚ.

In [20]:
let f := x ↦ 3x² + x + 1 in ℚ[x]

1:0: f := 3x^2 + x + 1 ∈ ℚ[x]


$d(f) = (6x + 1)\,dx$ — the universal differential, a 1-form. The $dx$
is part of the value; it's not just notation.

In [21]:
assert d(f) = (6x + 1) dx

1:0: ✓ d(f) = (6x + 1) dx


$(d/dx)(f) = 6x + 1$ — the derivation, a plain polynomial. Note the
absence of $dx$: this is the coefficient of the differential, not the
differential itself.

In [22]:
assert (d/dx)(f) = 6x + 1

1:0: ✓ (d/dx)(f) = 6x + 1


$\int f\,dx = x^3 + x^2/2 + x + \mathbb{Q}$. The $+ \mathbb{Q}$ is the
constant of integration, presented as a coset: any rational constant added
to $x^3 + x^2/2 + x$ is also an antiderivative. This is not a notational
convention — the system models the indefinite integral as a set.

In [23]:
∫ f dx

1:0: x^3 + (1/2)x^2 + x + ℚ


x^3 + (1/2)x^2 + x + ℚ

## 7 · Documented ceiling

Not every mathematically meaningful operation has a backend implementation
yet. When you ask for one, the system returns a **structured capability
gap** — it names the operation, the receiver, and the reason it can't
proceed. This is not a crash or a hidden method; it's an auditable entry
in the developer backlog.

Here, `det` is semantically available on $	ext{Mat}_2(\mathbb{Z}/5)$
(a matrix ring over a commutative ring always has a determinant), but no
backend has registered a realization for matrices over $\mathbb{Z}/5$
yet. Over ℚ it works (we just used it); over ℤ/5 it's a gap.

In [24]:
let N := [1, 2; 3, 4] in Mat₂(ℤ/5)

1:0: N := [1, 2; 3, 4] ∈ Mat₂(ℤ/5)


In [25]:
N.det()

LeanError: NoImplementation: 'det' is mathematically available here, but no registered route can execute it for this presentation.
  method:            det
  receiver category: MatrixElems(2, ℤ/5)
  presentation:      [1, 2; 3, 4] ∈ Mat₂(ℤ/5)
  semantic path:     declared directly on MatrixElems(2, ℤ/5)
  routes considered: 1
    - det for element of Mat(_, ℚ) → backend sage, op "mat_det_q", priority 0
This is a developer backlog item, not a narrowing of the mathematics: the method stays available on the category.